# 🥾 Side Trails · Step 5 — Software Stack, INT8 and Data Parallelism
**A Billion Small Steps · Hardware for AI · Step 5 · Book sections 5.6, 5.9 · Foothold lessons 5.6, 5.9**

Read the book sections first, then run this notebook from top to bottom. Runs on a **GPU**: Runtime → Change runtime type → **T4 GPU**.

| Part | What you do | Book |
|---|---|---|
| A | Every layer of the software stack, a device picker, real kernel names | 5.6 (leaves 5.6 → rejoin 5.7) |
| B | Shrink the MNIST model to INT8 | 5.9 (leaves 5.9 → rejoin 5.C) |
| C | Prove data parallelism on one GPU | 5.9 (leaves 5.9 → rejoin 5.C) |

⏱️ About 1–1½ hours. 📘 Textbook: Hwu, Kirk & El Hajj Ch 1 · D2L §13.5–13.6. Course site: https://aishugithub.github.io/course_GPU/

🥾 **Side Trails are optional.** Nothing in a later Core Lab needs them.

💬 **Every line of code has a comment** (after the `#`, or after `//` inside CUDA C). Read the comment, then the line it explains.

## ▶️ How to run this notebook

1. **Save your own copy first:** menu **File → Save a copy in Drive**. Work in that copy — the original is read-only, and your copy is what you submit.
2. **Run cells in order, top to bottom.** Click a grey code cell and press **Shift + Enter** (or the ▶ button on its left). Wait for it to finish (a ✓ and the output appear) before running the next one.
3. **Switch on the GPU:** menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**. (This lab needs it. If Colab says no GPU is available, wait a while and try again — the free tier has limits.)
4. **Lost or confused?** Menu **Runtime → Restart session and run all** starts everything again from a clean slate.
5. Cells marked **✏️ YOUR TURN** are exercises: change the code, run it again, and write your answer in the text cell below it (double-click a text cell to edit it).

## 0 · Setup — run these cells first

In [ ]:
# Which versions is this session running? The book's printed outputs used Colab's free Tesla T4
# with PyTorch 2.14.1 (CUDA 13.0 build) on 8 Nov 2026; small differences in your numbers are normal.
# If a cell fails after a Colab update: remove the "#" at the start of the next line, run this cell,
# then Runtime → Restart session (slow; only when needed).
# !pip install torch==2.14.1
import platform                                    # platform: tells us the Python version
import numpy, torch                                # NumPy and PyTorch, to read their version numbers
print("Python ", platform.python_version())        # the Python version
print("NumPy  ", numpy.__version__)                # the NumPy version
print("PyTorch", torch.__version__)                # the PyTorch version (and its CUDA build, after the +)
try:                                               # CuPy is only used by the CUDA C kernel labs
    import cupy                                    # try to load it
    print("CuPy   ", cupy.__version__)             # its version
except Exception:                                  # not installed, or no GPU for it in this session
    print("CuPy    not available in this session") # that is fine for this notebook
if torch.cuda.is_available():                      # is there an NVIDIA GPU in this session?
    print("GPU    ", torch.cuda.get_device_name(0))  # yes: its name (the book used a Tesla T4)
else:                                              # no GPU
    print("GPU     none (CPU runtime)")            # say so

In [ ]:
import time, subprocess            # time: stopwatch; subprocess: run system commands
import numpy as np                  # arrays and maths on the CPU
import matplotlib.pyplot as plt     # charts
import torch                        # PyTorch: runs maths on the CPU or the GPU
from torch import nn                # nn: PyTorch's layers and losses

if torch.cuda.is_available():       # is there an NVIDIA GPU we can use? ("cuda" = NVIDIA's GPU system)
    device = "cuda"                 # yes: from now on, device = "cuda" means "put it on the GPU"
    print("✅ GPU found:", torch.cuda.get_device_name(0))   # the name of GPU number 0 (the first one)
else:                               # no GPU in this session
    device = "cpu"                  # fall back to the CPU
    print("⚠️ NO GPU! Go to Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")   # how to get one
    print("   (Most of this notebook needs the GPU, so switch it on before going further.)")              # and why it matters

def sync():                         # GPU work runs in the background: wait for it to really finish before reading the clock
    if device == "cuda":            # (only needed when there is a GPU)
        torch.cuda.synchronize()    # wait until every queued GPU job is done

def timeit(fn, repeats=5):          # run fn several times and return the average time of one run, in seconds
    fn(); sync()                    # warm-up run (the first GPU call has extra start-up cost)
    t0 = time.perf_counter()        # start the stopwatch
    for _ in range(repeats):        # run it several times
        fn()                        # the work itself
    sync()                          # wait until the GPU has really finished all the runs
    return (time.perf_counter() - t0) / repeats   # total time ÷ number of runs

## Part A · The Software Stack  *(Section 5.6)*

Check every layer: PyTorch, the CUDA version it was built for, cuDNN, and the driver.

In [ ]:
print("PyTorch       :", torch.__version__)   # the PyTorch layer
print("Built for CUDA:", torch.version.cuda)   # the CUDA version PyTorch was built for (None = a CPU-only build)
print("cuDNN         :", torch.backends.cudnn.version())   # NVIDIA's neural-network library
print("GPU available :", torch.cuda.is_available())   # can PyTorch actually reach a GPU?
# the driver layer: ask nvidia-smi for the GPU's name and the NVIDIA driver's version
!nvidia-smi --query-gpu=name,driver_version --format=csv

**A portable device picker:** the same code runs on NVIDIA, Apple or CPU.

In [ ]:
if torch.cuda.is_available():   # an NVIDIA GPU (or an AMD one with ROCm)?
    dev = "cuda"   # use it
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():   # an Apple M-series GPU?
    dev = "mps"   # use that
else:   # otherwise...
    dev = "cpu"   # ...the CPU
print("Using:", dev)   # show the choice

**Real kernel names:** profile a small model and look for GEMM (matrix-multiply) kernels, element-wise kernels and copies.

In [ ]:
from torch.profiler import profile, ProfilerActivity   # PyTorch's built-in profiler
net = nn.Sequential(nn.Linear(1024, 4096), nn.ReLU(), nn.Linear(4096, 10)).to(device)   # a small model on the GPU
x_cpu = torch.randn(256, 1024)   # a batch that starts on the CPU, so we also see a copy
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:   # record the CPU and the GPU
    for _ in range(5):   # five forward passes
        y = net(x_cpu.to(device))   # copy the batch over, then run the model
    sync()   # wait for the GPU before the recording stops
gpu_rows = [e for e in prof.key_averages() if getattr(e, "self_device_time_total", 0) > 0]   # the entries that ran on the GPU
for e in sorted(gpu_rows, key=lambda e: -e.self_device_time_total)[:6]:   # the 6 that took the most GPU time
    print(f"{e.key[:58]:58s} {e.self_device_time_total:7.0f} µs")   # its name and GPU time in microseconds

### ✏️ YOUR TURN (A)
1. Write down the PyTorch, CUDA and driver versions. Is the driver new enough for the CUDA build (Section 5.6's rule)?
2. In the profiler list, find one GEMM kernel, one element-wise (ReLU) kernel and one memory copy. Copy their names here.

*Your answers:*

---
## Part B · Shrink the MNIST model to INT8  *(Section 5.9)*
Train the Core Lab 3 network (784 → 128 → 10, 3 epochs), then store every weight as an 8-bit whole number plus one scale factor per layer, and compare size, speed and accuracy. INT8 here runs on the **CPU**, as on a phone or an edge board. (About 1 minute.)

In [ ]:
import io, os                                    # io: an in-memory "file"; os: count the CPU cores
from torchvision import datasets, transforms     # MNIST, and a converter from pictures to tensors
torch.manual_seed(0)                             # a fixed random start, so runs match
torch.set_num_threads(min(4, os.cpu_count() or 1))   # Colab has 2 CPU cores; use at most 4
tf = transforms.ToTensor()                       # pixels 0..255 → floats 0..1
train = datasets.MNIST("data", train=True, download=True, transform=tf)    # 60,000 training images
test = datasets.MNIST("data", train=False, download=True, transform=tf)    # 10,000 test images
train_loader = torch.utils.data.DataLoader(train, batch_size=64, shuffle=True)   # batches of 64
X_test = torch.stack([test[i][0] for i in range(len(test))])   # all 10,000 test pictures in one tensor
y_test = torch.tensor([test[i][1] for i in range(len(test))])  # and their right answers

model = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))   # the Core Lab 3 network
opt = torch.optim.SGD(model.parameters(), lr=0.1)   # gradient descent, learning rate 0.1
loss_fn = nn.CrossEntropyLoss()                  # the loss for "which digit?"
for epoch in range(3):                           # 3 epochs, on the CPU
    for x, y in train_loader:                    # one batch at a time
        loss = loss_fn(model(x), y)              # 1. predict and 2. measure the loss
        opt.zero_grad(); loss.backward(); opt.step()   # 3. slopes, 4. nudge every weight
model.eval()                                     # finished training: switch to "using" mode

def size_kb(m):                                  # the size of a model's saved weights, in KB
    buf = io.BytesIO()                           # an empty in-memory file
    torch.save(m.state_dict(), buf)              # save the weights into it
    return buf.tell() / 1024                     # its length in bytes ÷ 1,024

def accuracy_and_ms(m):                          # accuracy on 10,000 test images, and the time for them
    with torch.no_grad():                        # only using the model, not training
        m(X_test)                                # a warm-up run
        best = float("inf")                      # the best of 20 timed runs, in ms
        for _ in range(20):                      # 20 runs (Colab's 2 cores make single timings jumpy)
            t = time.perf_counter()              # start the stopwatch
            pred = m(X_test).argmax(1)           # the guess for every test image
            best = min(best, (time.perf_counter() - t) * 1000)   # keep the fastest
    return (pred == y_test).float().mean().item(), best   # fraction right, and ms
print("trained; threads used:", torch.get_num_threads())   # ready

In [ ]:
# INT8: each nn.Linear keeps its weights as 8-bit whole numbers and a scale factor
try:
    # the classic PyTorch way (prints a DeprecationWarning on recent versions, but works)
    qmodel = torch.ao.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)
# if your PyTorch has removed it, use its replacement, the torchao library
except (AttributeError, RuntimeError, NotImplementedError):
    # torchao: PyTorch's new home for quantisation (on Colab: !pip install torchao)
    from torchao.quantization import quantize_, Int8DynamicActivationInt8WeightConfig
    # a copy of the model, so the FP32 one stays as it is
    import copy; qmodel = copy.deepcopy(model)
    # turn every nn.Linear's weights into INT8, in place
    quantize_(qmodel, Int8DynamicActivationInt8WeightConfig())
# the same test for both models: size, time for 10,000 images, accuracy
for name, m in [("FP32", model), ("INT8", qmodel)]:
    # measure this model
    acc, ms = accuracy_and_ms(m)
    # print one line per model
    print(f"{name}: size {size_kb(m):6.1f} KB   10,000 test images in {ms:5.1f} ms   accuracy {acc:.2%}")

### ✏️ YOUR TURN (B)
| | FP32 | INT8 |
|---|---|---|
| Size (KB) | | |
| Time for 10,000 images (ms) | | |
| Accuracy | | |

1. The size dropped to about a quarter. Why a quarter? (Section 5.2: bytes per number.)
2. Did the accuracy change? Would you ship the INT8 model on a phone?

*Your answers:*

## Part C · Prove Data Parallelism  *(Section 5.9)*

Four GPUs each take a quarter of the batch, compute gradients, and average them (all-reduce). Here we do it with ONE device: split the batch into 4 slices, compute each slice's gradient, average, and compare with the full-batch gradient.

In [ ]:
torch.manual_seed(0)   # the same starting weights every time
net = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10)).to(device)   # a small network
X = torch.randn(256, 784, device=device); Y = torch.randint(0, 10, (256,), device=device)   # one batch of 256

net.zero_grad()   # one "GPU" with the whole batch: clean slopes...
nn.functional.cross_entropy(net(X), Y).backward()   # ...forward and backward on all 256
full = [p.grad.clone() for p in net.parameters()]   # save a copy of every parameter's slope

slice_grads = []   # four "GPUs", 64 examples each
for xs, ys in zip(X.chunk(4), Y.chunk(4)):   # .chunk(4) cuts the batch into 4 equal slices
    net.zero_grad()   # each "GPU" starts with clean slopes
    nn.functional.cross_entropy(net(xs), ys).backward()   # forward and backward on its slice
    slice_grads.append([p.grad.clone() for p in net.parameters()])   # keep its slopes
averaged = [sum(g[i] for g in slice_grads) / 4 for i in range(len(full))]   # the all-reduce: add the 4 slices' slopes, divide by 4

diff = max((a - b).abs().max().item() for a, b in zip(full, averaged))   # the biggest difference anywhere
print(f"Largest difference between full-batch and averaged gradients: {diff:.2e}")   # show it
print("Identical (up to rounding): data parallelism computes the SAME update." if diff < 1e-5 else "Check the code.")   # the verdict

### ✏️ YOUR TURN (C)
1. Why does averaging work only when the slices are the same size? (Hint: what does `cross_entropy` do by default: sum or mean?)
2. In real data parallelism the averaging happens over NVLink. Using Amdahl's law, why must that exchange be fast?

*Your answers:*

---
## 🎓 Where next?
Back to the Main Climb at Section 5.7 / 5.C (Core Lab 12), or on to the **GPUs Beyond AI** Side Trail notebook.